# XR Client Utilities

Examples of defining user commands that can be invoked by the XR client's command menu.

<video controls src="assets/xr-client-utilities.mp4" alt="video demonstrating the commands in xr" />

## Setup runner

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="xr client utilities example")
imd_runner.load(0)

## Runner utilities
You can wrap a set of utilities around the runner to add extra commands:

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## XR client notification
The XR client defines a remote command to send it notifications (displayed over the controller); the utilities provide a convenience method `notify_all` to call all defined commands of this type.

We define an additional local notification command:

In [3]:
def test_notify(message):
    print(message)

imd_runner.app_server.register_command("test/notify", test_notify)

The `notify_all` triggers the notification command of any connected XR clients, in addition to the command we just defined:

In [4]:
utilities.notify_all("TESTING!")

TESTING!


## Interaction modes

The utilities provide a basic system of interaction "modes" that define user commands for switching between different custom behaviors in response to iMD interactions:

In [5]:
utilities.use_interaction_modes()

Defining a mode where every user interaction triggers a notification in the XR client with information about the interacted atom from MDAnalysis:

In [6]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from nanover.mdanalysis import frame_data_to_mdanalysis

# prepare info from simulation
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

class InfoMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        atom = universe.atoms[interaction.particles[0]]
        utilities.notify_all(str(atom))

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass

utilities.add_interaction_mode(InfoMode, "info")

## Recording
The utilities define user commands for starting and stopping recordings on the fly:

In [7]:
utilities.use_recording_commands()

In [8]:
imd_runner.app_server.run_command("commands/list", {}).result()

{'list': {'commands/list': {},
  'multiuser/radially-orient-origins': {},
  'playback/load': {},
  'playback/next': {},
  'playback/list': {},
  'playback/reset': {},
  'playback/pause': {},
  'playback/play': {},
  'playback/step': {},
  'test/notify': {},
  'user/interaction/normal': {},
  'user/interaction/info': {},
  'user/recording/start': {},
  'user/recording/stop': {},
  'user/recording/checkpoint': {}}}